In [1]:
library(ipumsr)
library(dplyr)
library(survey)
library(srvyr)


Attaching package: ‘dplyr’




The following objects are masked from ‘package:stats’:

    filter, lag




The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




Loading required package: grid



Loading required package: Matrix



Loading required package: survival




Attaching package: ‘survey’




The following object is masked from ‘package:graphics’:

    dotchart





Attaching package: ‘srvyr’




The following object is masked from ‘package:stats’:

    filter




In [2]:
#ddi <- read_ipums_ddi("usa_00064.xml")
data <- read_ipums_micro(data_file = "usa_00064.dat", ddi = "usa_00064.xml")

Use of data from IPUMS USA is subject to conditions including that users should cite the data appropriately. Use command `ipums_conditions()` for more details.



In [3]:
#Reno <- read.csv(file='usa_00063.csv', header=TRUE)
#head(data)

Reno does not appear in the 1% samples prior to 1960, pulling full data for 1850-1950. Because this creates a large extract, data now filtered in retrieval. Retrieval logic does not allow combining multiple metro case selectors, stitching together two extracts vertically to accomodate filters for 1850-2000-METAREA and 2010-2024 MET2013.

In [4]:
#data <- data %>% filter(METAREA==672 | MET2013==39900)

In [5]:
#glimpse(data)

Fix missing

In [6]:
#IpumsR import does not code R native (NA) missing values
sum(is.na(data$RACE))
sum(is.na(data$HISPAN))

[1] 0

[1] 0

In [11]:
#RACE is harmonized by IPUMS to not include missing values, it is inferred from other household members. The single multilevel race variable RACHSING is not available before 2000. However, HISPAN explicitly codes a missing value.
#ipums_val_labels(data$RACE)
#ipums_val_labels(data$HISPAN)

In [9]:
#data %>% group_by(HISPAN=haven::as_factor(HISPAN)) %>% summarize(n=sum(PERWT)) %>% mutate(pct = n/sum(n))
HISPAN <- lbl_na_if (data$HISPAN, ~ .val == 9)
data %>% group_by(HISPAN=haven::as_factor(HISPAN)) %>% summarize(n=sum(PERWT)) %>% mutate(pct = n/sum(n))

In [5]:
data %>% group_by(RACE=haven::as_factor(RACE)) %>% summarize(n=sum(PERWT)) %>% mutate(pct = n/sum(n))

RACE,n,pct
<fct>,<dbl>,<dbl>
White,1492046,0.772383283
Black/African American,48610,0.025163803
American Indian or Alaska Native,32982,0.017073700
Chinese,18236,0.009440179
Japanese,7179,0.003716333
Other Asian or Pacific Islander,79037,0.040914863
"Other race, nec",128182,0.066355618
Two major races,115287,0.059680299
Three or more major races,10184,0.005271923


In [6]:
#ipums_val_labels(data$RACE)
#ipums_var_label(data$RACE)
#ipums_var_info(data$RACE)

In [7]:
#lbl_na_if()
#lbl_clean()
#as_factor()
#zap_labels()
#zap_ipums_attributes()

Recode using dplyr (Tidyverse according to rpubs.com).
We code two integrated race ethnicity variables, RACEETH1 is White-inclusive for Latino, meaning those Latinos selecting White race are coded White. RACEETH2 reverses this logic, coding those individuals responding White as Latino if they identified ethnicity as Hispanic.

In [13]:
data <- data %>%
    mutate(RACEETH1 = case_when(
        RACE >6 & RACE <=9 ~ "Other",
        HISPAN >1 & HISPAN <5 ~ "Latino",
        RACE == 1 ~ "White",
        RACE >3 & RACE <7 ~ "Asian"
        RACE == 2 ~ "Black",
        RACE == 3 ~ "American Indian",
        ))


In [13]:
data <- data %>%
    mutate(RACEETH2 = case_when(
        RACE >6 & RACE <=9 ~ "Other",
        RACE == 1 ~ "White",
        RACE >3 & RACE <7 ~ "Asian"
        HISPAN >1 & HISPAN <5 ~ "Latino",
        RACE == 2 ~ "Black",
        RACE == 3 ~ "American Indian",
        ))


In [15]:
#Summarize RACEETH1 and RACEETH2 across all samples in the dataset
data %>% group_by (RACEETH1) %>% summarize(n=sum(PERWT)) %>% mutate(pct= n/sum(n))
data %>% group_by (RACEETH2) %>% summarize(n=sum(PERWT)) %>% mutate(pct= n/sum(n))

RACEETH,n,pct
<chr>,<dbl>,<dbl>
American Indian,30922,0.01600731
Asian,103448,0.05355164
Black,47308,0.02448980
Latino,44112,0.02283534
Other,253653,0.13130784
White,1452300,0.75180808


In [17]:
#Recode IPUMS SAMPLE into DECADE
#ipums_val_labels(data$SAMPLE)
data <- data %>%
    mutate(DECADE = case_when(
        SAMPLE >= 185000 & SAMPLE < 186000 ~ "1850",
        SAMPLE >= 186000 & SAMPLE < 187000 ~ "1860",
        SAMPLE >= 187000 & SAMPLE < 188000 ~ "1870",
        SAMPLE >= 188000 & SAMPLE < 189000 ~ "1880",
        SAMPLE >= 189000 & SAMPLE < 190000 ~ "1890",
        SAMPLE >= 190000 & SAMPLE < 191000 ~ "1900",
        SAMPLE >= 191000 & SAMPLE < 192000 ~ "1910",
        SAMPLE >= 192000 & SAMPLE < 193000 ~ "1920",
        SAMPLE >= 193000 & SAMPLE < 194000 ~ "1930",
        SAMPLE >= 194000 & SAMPLE < 195000 ~ "1940",
        SAMPLE >= 195000 & SAMPLE < 196000 ~ "1950",
        SAMPLE >= 196000 & SAMPLE < 197000 ~ "1960",
        SAMPLE >= 197000 & SAMPLE < 198000 ~ "1970",
        SAMPLE >= 198000 & SAMPLE < 199000 ~ "1980",
        SAMPLE >= 199000 & SAMPLE < 200000 ~ "1990",
        SAMPLE >= 200000 & SAMPLE < 201000 ~ "2000",
        SAMPLE >= 201000 & SAMPLE < 202000 ~ "2010",
        SAMPLE >= 202000 & SAMPLE < 203000 ~ "2020",
        ))
data %>% group_by (DECADE) %>% summarize(n=sum(PERWT)) %>% mutate(pct = n/sum(n))

DECADE,n,pct
<chr>,<dbl>,<dbl>
1960,84700,0.04384641
1980,190400,0.09856384
1990,255866,0.13245344
2010,893909,0.46274737
2020,506868,0.26238894


In [14]:
#crosstab over time
data %>% group_by(SAMPLE, RACETH1) %>%
    tally() %>%
    spread(SAMPLE, n)

ERROR: Error in spread(., SAMPLE, n): could not find function "spread"


In [16]:
summary(data)

      YEAR          SAMPLE           SERIAL           CBSERIAL        
 Min.   :1960   Min.   :196002   Min.   : 313446   Min.   :2.396e+03  
 1st Qu.:1980   1st Qu.:198002   1st Qu.: 763358   1st Qu.:1.228e+06  
 Median :2010   Median :201001   Median : 779439   Median :2.019e+12  
 Mean   :2001   Mean   :200151   Mean   : 898832   Mean   :1.439e+12  
 3rd Qu.:2019   3rd Qu.:201901   3rd Qu.: 831444   3rd Qu.:2.024e+12  
 Max.   :2024   Max.   :202401   Max.   :1652697   Max.   :2.024e+12  
                                                   NA's   :8541       
      HHWT           CLUSTER             METAREA         METAREAD    
 Min.   :  0.00   Min.   :1.960e+12   Min.   :672     Min.   :6720   
 1st Qu.: 45.00   1st Qu.:1.980e+12   1st Qu.:672     1st Qu.:6720   
 Median : 71.00   Median :2.010e+12   Median :672     Median :6720   
 Mean   : 83.67   Mean   :2.002e+12   Mean   :672     Mean   :6720   
 3rd Qu.:100.00   3rd Qu.:2.019e+12   3rd Qu.:672     3rd Qu.:6720   
 Max.   :699

In [8]:
#save(Reno_clean, file="Reno.RData")
saveRDS(data, file = "Reno.rds")